# Schematics NPC DSC 2026 — Bird-Sound Classification

**Model:** EfficientNet-style CNN **trained from scratch** on the competition
audio only (random initialisation, no pretrained weights, no external data).
**Classes:** Bondol Jawa, Cendrawasih, Kutilang, + open-set **Unknown**.
**Metric:** Macro-F1 over the four classes.

### How it works (fully explainable, per competition rules)
1. **Front-end** — 32 kHz mono → PCEN log-mel spectrogram, standardised per clip.
   PCEN suppresses stationary background; per-clip norm removes recording-level
   differences so loudness can't leak the source.
2. **Validation** — `StratifiedGroupKFold` on `source_group`. Every source group
   contains one species, so whole groups are held out — no recording leaks
   between train and validation. A 300-seed search picks the most class-balanced
   split (Cendrawasih has only 4 groups).
3. **Training** — activity-weighted 5 s crops + **cross-group background mixing**
   (the biggest known gain in bird-sound tasks) + mild EQ/gain + SpecAugment.
   Class-balanced sampler and √-inverse-frequency loss for the 13.5:1 imbalance,
   with light label smoothing (also calibrates the Unknown signal).
4. **Unknown** — a clip is **Unknown** when the ensemble's top probability is
   below a threshold τ. τ is chosen by **leave-one-class-out** (hold a species
   out entirely as a stand-in for an unseen one) — never using the test set.
5. **Inference** — slide 5 s windows (50 % overlap) over each test clip, average
   softmax over windows and the K fold models, apply τ.

> Reproducibility: one `SEED` controls numpy, torch, CUDA and the sampler.
> Set `FAST=True` for a quick dry run; `False` for the full submission.

## 0. Config

In [ ]:
import os, sys, json, time, math, hashlib, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
FAST = False          # True = 1 fold / few epochs for a smoke run
N_FOLDS = 4
EPOCHS = 22 if not FAST else 2
BATCH = 32
SIZE = "s"            # xs | s | m  (CNN width)
FRONTEND = "pcen_norm"
AUTHOR = "Arif"       # submission file name -> {AUTHOR}_{SUBMISSION_NO:02d}_Submission.csv
SUBMISSION_NO = 1     # bump for every new upload (Arif_01 -> Arif_02 ...)

# ---- audio / spectrogram ----
ORIG_SR, SR = 48_000, 32_000
WIN_SEC, EVAL_HOP_SEC = 5.0, 2.5
WIN = int(SR * WIN_SEC)
N_FFT, HOP, N_MELS, FMIN, FMAX = 1024, 320, 128, 40, 15_000
PCEN = dict(gain=0.8, bias=10.0, power=0.25, time_constant=0.06, eps=1e-6)
ACT_FMIN, ACT_FMAX, ACT_SMOOTH_SEC = 500, 12_000, 0.25
NOISE_MIN_SEC, NOISE_QUANTILE, NOISE_PER_CLIP = 1.0, 0.30, 2
FOLD_SEED_SEARCH = 300

CLASSES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
UNKNOWN = "Unknown"
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# ---- paths: find the Kaggle dataset automatically ----
def find_data():
    for base in ["/kaggle/input", "."]:
        for p in Path(base).rglob("sample_submission.csv"):
            return p.parent
    raise FileNotFoundError("sample_submission.csv not found under /kaggle/input")
DATA = find_data()
TRAIN_AUDIO, TEST_AUDIO = DATA/"train"/"audio", DATA/"test"/"audio"
TRAIN_META, SAMPLE_SUB = DATA/"train"/"metadata.csv", DATA/"sample_submission.csv"
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
CACHE = OUT/"cache"; (CACHE/"audio"/"train").mkdir(parents=True, exist_ok=True)
(CACHE/"audio"/"test").mkdir(parents=True, exist_ok=True)
(CACHE/"activity").mkdir(parents=True, exist_ok=True)
(CACHE/"noise").mkdir(parents=True, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print("data:", DATA)
print("train clips:", len(list(TRAIN_AUDIO.glob('*.flac'))),
      "| test clips:", len(list(TEST_AUDIO.glob('*.flac'))))

## 1. Front-end — PCEN log-mel + per-clip norm, and the activity curve

In [ ]:
import librosa, soundfile as sf
_MEL_KW = dict(sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmin=FMIN, fmax=FMAX, center=True)

def load_mono_32k(path):
    y, sr = sf.read(path, dtype="float32", always_2d=True)
    y = y.mean(1)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR, res_type="soxr_hq")
    return np.ascontiguousarray(y, dtype=np.float32)

def pcen_spec(y):
    S = librosa.feature.melspectrogram(y=y, power=1.0, **_MEL_KW)
    P = librosa.pcen(S * (2**31), sr=SR, hop_length=HOP, **PCEN)
    return P.astype(np.float32)

def clip_norm(X):
    return ((X - X.mean()) / (X.std() + 1e-6)).astype(np.float32)

def frontend(y):                       # pcen_norm
    return clip_norm(pcen_spec(y))

def activity_curve(y):
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                       n_mels=64, fmin=ACT_FMIN, fmax=ACT_FMAX, power=2.0)
    D = librosa.power_to_db(S, ref=1.0, amin=1e-10)
    excess = np.clip(D - np.median(D, axis=1, keepdims=True), 0, None).mean(0)
    k = max(1, int(ACT_SMOOTH_SEC * SR / HOP))
    return np.convolve(excess, np.ones(k)/k, mode="same").astype(np.float32)

## 2. Cache audio + activity (one decode per clip, reused every epoch)

In [ ]:
from joblib import Parallel, delayed

def cache_one(path, split):
    fid = path.stem
    ap = CACHE/"audio"/split/f"{fid}.npy"
    if not ap.exists():
        y = load_mono_32k(path); np.save(ap, y)
    else:
        y = np.load(ap)
    if split == "train":
        cp = CACHE/"activity"/f"{fid}.npy"
        if not cp.exists(): np.save(cp, activity_curve(y))
    return fid

train_files = sorted(TRAIN_AUDIO.glob("*.flac"))
test_files  = sorted(TEST_AUDIO.glob("*.flac"))
if FAST:
    train_files = train_files[:120]; test_files = test_files[:40]

t0 = time.time()
Parallel(n_jobs=os.cpu_count(), backend="threading")(
    delayed(cache_one)(p, "train") for p in train_files)
Parallel(n_jobs=os.cpu_count(), backend="threading")(
    delayed(cache_one)(p, "test") for p in test_files)
print(f"cached {len(train_files)} train + {len(test_files)} test in {time.time()-t0:.0f}s")

## 3. Group folds — hold out whole source groups, balanced-split seed search

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

meta = pd.read_csv(TRAIN_META)
meta = meta[meta.file_id.isin([p.stem for p in train_files])].reset_index(drop=True)
meta["y"] = meta.label.map(CLASS_TO_IDX)

def best_group_folds(meta, n_folds=N_FOLDS, n_seeds=FOLD_SEED_SEARCH):
    y, g = meta.y.values, meta.source_group.values
    best, best_seed, best_fold = -1, None, None
    for s in range(n_seeds):
        sgkf = StratifiedGroupKFold(n_folds, shuffle=True, random_state=s)
        fold = np.full(len(meta), -1)
        for k, (_, va) in enumerate(sgkf.split(meta, y, g)):
            fold[va] = k
        # score = how evenly each class is spread across folds (want all present)
        ok = all(len(set(y[fold == k])) == len(CLASSES) for k in range(n_folds))
        spread = min(np.bincount(fold[y == c], minlength=n_folds).min() for c in range(len(CLASSES)))
        score = (1000 if ok else 0) + spread
        if score > best:
            best, best_seed, best_fold = score, s, fold.copy()
    return best_fold, best_seed, best

fold, fseed, fscore = best_group_folds(meta)
meta["fold"] = fold
meta.to_csv(OUT/"folds.csv", index=False)
print(f"fold seed={fseed} score={fscore}")
print(pd.crosstab(meta.fold, meta.label))

## 4. Noise bank — quiet, call-free segments for cross-group mixing

In [ ]:
def build_noise_bank(meta):
    rows = []
    fps = SR / HOP
    seg_len = int(NOISE_MIN_SEC * SR)
    step = max(1, int(0.5 * fps))
    for _, r in meta.iterrows():
        y = np.load(CACHE/"audio"/"train"/f"{r.file_id}.npy")
        curve = np.load(CACHE/"activity"/f"{r.file_id}.npy")
        quiet = curve <= np.quantile(curve, NOISE_QUANTILE)
        got = 0
        for f0 in range(0, max(1, len(curve) - int(NOISE_MIN_SEC*fps)), step):
            f1 = f0 + int(NOISE_MIN_SEC*fps)
            if quiet[f0:f1].mean() > 0.8:
                seg = y[int(f0*HOP): int(f0*HOP)+seg_len]
                if len(seg) >= seg_len:
                    nf = f"{r.file_id}_{got}.npy"; np.save(CACHE/"noise"/nf, seg.astype(np.float32))
                    rows.append(dict(noise_file=nf, source_group=r.source_group)); got += 1
                    if got >= NOISE_PER_CLIP: break
    nb = pd.DataFrame(rows); nb.to_csv(OUT/"noise_bank.csv", index=False)
    return nb
noise_bank = build_noise_bank(meta)
print("noise segments:", len(noise_bank))

## 5. Augmentation + Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

def loop_pad(y, n):
    if len(y) >= n: return y
    return np.tile(y, int(np.ceil(n/max(len(y),1))))[:n]

def window_scores(curve, n, stride_sec=0.25):
    win_f = max(1, round(n/HOP)); stride_f = max(1, round(stride_sec*SR/HOP))
    if len(curve) <= win_f: return np.array([0]), np.array([curve.mean() if len(curve) else 0.0])
    cs = np.concatenate([[0.0], np.cumsum(curve, dtype=np.float64)])
    st = np.arange(0, len(curve)-win_f+1, stride_f)
    return st*HOP, (cs[st+win_f]-cs[st])/win_f

def choose_crop(y, curve, n, rng, p_random=0.2, temp=0.5):
    y = loop_pad(y, n); m = len(y)-n
    if m <= 0: return y[:n]
    if curve is None or rng.random() < p_random:
        s = int(rng.integers(0, m+1)); return y[s:s+n]
    st, sc = window_scores(curve, n); st = np.minimum(st, m)
    z = (sc-sc.max())/(sc.std()+1e-6)/temp; p = np.exp(z); p/=p.sum()
    s = int(rng.choice(st, p=p)); return y[s:s+n]

def best_crop(y, curve, n):
    y = loop_pad(y, n)
    if curve is None or len(y) <= n: return y[:n]
    st, sc = window_scores(curve, n); return y[min(int(st[int(np.argmax(sc))]), len(y)-n):][:n]

def eval_windows(y, n=WIN, hop=int(SR*EVAL_HOP_SEC)):
    if len(y) <= n: return [loop_pad(y, n)]
    st = list(range(0, len(y)-n+1, hop))
    if st[-1] != len(y)-n: st.append(len(y)-n)
    return [y[s:s+n] for s in st]

def rms(x): return float(np.sqrt(np.mean(x**2, dtype=np.float64))+1e-12)

class NoiseBank:
    def __init__(self, nb): self.nb = nb.reset_index(drop=True); self._c = {}
    def __len__(self): return len(self.nb)
    def get(self, i):
        if i not in self._c: self._c[i] = np.load(CACHE/"noise"/self.nb.at[i,"noise_file"]).astype(np.float32)
        return self._c[i]
    def sample(self, n, rng, exclude_group=None):
        idx = np.arange(len(self.nb))
        if exclude_group is not None: idx = idx[self.nb.source_group.values != exclude_group]
        if len(idx) == 0: return None
        x = loop_pad(self.get(int(rng.choice(idx))), n); s = int(rng.integers(0, len(x)-n+1)); return x[s:s+n]

def mix_noise(y, nz, snr):
    return (y + rms(y)/(rms(nz)*10**(snr/20))*nz).astype(np.float32)

def random_eq(y, rng, max_db=4.0, k=6):
    Y = np.fft.rfft(y); f = np.fft.rfftfreq(len(y), 1/SR)
    knots = np.geomspace(50, SR/2, k); g = 10**(np.interp(np.log(np.maximum(f,50)), np.log(knots), rng.uniform(-max_db,max_db,k))/20)
    return np.fft.irfft(Y*g, n=len(y)).astype(np.float32)

def spec_augment(S, rng, n_t=2, max_t=40, n_f=2, max_f=12):
    S = S.copy(); fill = S.mean(); Fb, T = S.shape
    for _ in range(n_f):
        w=int(rng.integers(0,max_f+1)); a=int(rng.integers(0,max(1,Fb-w))); S[a:a+w]=fill
    for _ in range(n_t):
        w=int(rng.integers(0,max_t+1)); a=int(rng.integers(0,max(1,T-w))); S[:,a:a+w]=fill
    return S

class BirdDS(Dataset):
    def __init__(self, table, train=True, noise=None, seed=SEED, split="train"):
        self.t = table.reset_index(drop=True); self.train=train; self.noise=noise
        self.rng = np.random.default_rng(seed); self.split=split
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        r = self.t.iloc[i]
        y = np.load(CACHE/"audio"/self.split/f"{r.file_id}.npy").astype(np.float32)
        if self.train:
            cp = CACHE/"activity"/f"{r.file_id}.npy"
            curve = np.load(cp).astype(np.float32) if Path(cp).exists() else None
            w = choose_crop(y, curve, WIN, self.rng)
            if self.noise is not None and len(self.noise) and self.rng.random() < 0.8:
                for _ in range(int(self.rng.integers(1,3))):
                    nz = self.noise.sample(len(w), self.rng, exclude_group=r.source_group)
                    if nz is not None: w = mix_noise(w, nz, self.rng.uniform(3,20))
            if self.rng.random() < 0.5: w = random_eq(w, self.rng)
            if self.rng.random() < 0.5: w = w*10**(self.rng.uniform(-6,6)/20)
            w = np.clip(w, -1, 1)
            S = frontend(w)
            if self.rng.random() < 0.5: S = spec_augment(S, self.rng)
        else:
            cp = CACHE/"activity"/f"{r.file_id}.npy"
            curve = np.load(cp).astype(np.float32) if Path(cp).exists() else None
            S = frontend(best_crop(y, curve, WIN))
        lbl = int(r.y) if "y" in r and not pd.isna(r.y) else -1
        return S[None].astype(np.float32), lbl

## 6. Model — small EfficientNet-style CNN (from scratch)

In [ ]:
import torch.nn as nn
def swish(x): return x*torch.sigmoid(x)
class SE(nn.Module):
    def __init__(s,c,r=4): super().__init__(); s.f1=nn.Conv2d(c,max(1,c//r),1); s.f2=nn.Conv2d(max(1,c//r),c,1)
    def forward(s,x): w=x.mean((2,3),keepdim=True); return x*torch.sigmoid(s.f2(swish(s.f1(w))))
class MBConv(nn.Module):
    def __init__(s,ci,co,stride=1,e=4,k=3,drop=0.):
        super().__init__(); m=ci*e; s.res=stride==1 and ci==co
        s.exp=nn.Sequential(nn.Conv2d(ci,m,1,bias=False),nn.BatchNorm2d(m)) if e!=1 else None
        s.dw=nn.Conv2d(m,m,k,stride,k//2,groups=m,bias=False); s.bnd=nn.BatchNorm2d(m); s.se=SE(m)
        s.pw=nn.Conv2d(m,co,1,bias=False); s.bnp=nn.BatchNorm2d(co); s.drop=nn.Dropout2d(drop) if drop>0 else None
    def forward(s,x):
        h=x if s.exp is None else swish(s.exp(x)); h=swish(s.bnd(s.dw(h))); h=s.se(h); h=s.bnp(s.pw(h))
        if s.drop is not None: h=s.drop(h)
        return h+x if s.res else h
class BirdCNN(nn.Module):
    def __init__(s,n=3,width=(16,24,40,80,112),drop=0.1):
        super().__init__(); c0=width[0]
        s.stem=nn.Sequential(nn.Conv2d(1,c0,3,2,1,bias=False),nn.BatchNorm2d(c0))
        stages=[(width[1],1,1,3,1),(width[2],2,4,3,2),(width[3],2,4,5,2),(width[4],2,6,5,3)]
        b=[]; ci=c0
        for co,st,e,k,rep in stages:
            for i in range(rep): b.append(MBConv(ci,co,st if i==0 else 1,e,k,drop)); ci=co
        s.blocks=nn.Sequential(*b)
        s.head=nn.Sequential(nn.Conv2d(ci,ci*2,1,bias=False),nn.BatchNorm2d(ci*2))
        s.dp=nn.Dropout(drop); s.fc=nn.Linear(ci*2,n); s.embed_dim=ci*2
    def features(s,x): x=swish(s.stem(x)); x=s.blocks(x); return swish(s.head(x)).mean((2,3))
    def forward(s,x,emb=False):
        e=s.features(x); o=s.fc(s.dp(e)); return (o,e) if emb else o
def build_model(n=3,size=SIZE,drop=0.1):
    W={"xs":(12,16,24,48,64),"s":(16,24,40,80,112),"m":(24,32,56,112,160)}[size]
    return BirdCNN(n,W,drop)

## 7. Train the group-fold ensemble

In [ ]:
from sklearn.metrics import f1_score, recall_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def softmax(z): z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)

def make_loader(tbl, train, noise, seed):
    ds = BirdDS(tbl, train=train, noise=noise, seed=seed)
    if train:
        cnt = np.bincount(tbl.y.values, minlength=len(CLASSES)).astype(float)
        w = 1.0/cnt[tbl.y.values]
        sm = WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double), len(tbl), True)
        return DataLoader(ds, BATCH, sampler=sm, num_workers=2, drop_last=True, pin_memory=torch.cuda.is_available())
    return DataLoader(ds, BATCH, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

@torch.no_grad()
def infer_loader(model, loader):
    model.eval(); L=[]; E=[]; Y=[]
    for X,y in loader:
        o,e = model(X.to(device), emb=True)
        L.append(o.float().cpu().numpy()); E.append(e.float().cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(L), np.concatenate(E), np.concatenate(Y)

def train_fold(k, seed=SEED):
    tr = meta[meta.fold!=k].reset_index(drop=True); va = meta[meta.fold==k].reset_index(drop=True)
    nb = NoiseBank(noise_bank[noise_bank.source_group.isin(set(tr.source_group))])
    tl = make_loader(tr, True, nb, seed+k); vl = make_loader(va, False, None, seed+k)
    model = build_model().to(device)
    cnt = np.bincount(tr.y.values, minlength=len(CLASSES)).astype(float)
    cw = torch.as_tensor((cnt.sum()/(len(cnt)*cnt))**0.5, dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = max(1,len(tl))*EPOCHS
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type=="cuda")
    best, best_state = -1, None
    for ep in range(EPOCHS):
        model.train()
        for X,y in tl:
            X=X.to(device); y=y.to(device); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=device.type=="cuda"): loss=crit(model(X),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        lo,_,yv = infer_loader(model, vl); f1=f1_score(yv, lo.argmax(1), average="macro")
        if f1>=best: best=f1; best_state={kk:v.detach().cpu().clone() for kk,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    torch.save({"state_dict":best_state,"size":SIZE,"classes":CLASSES}, OUT/f"model_fold{k}.pt")
    lo,em,yv = infer_loader(model, vl)
    print(f"fold{k}: best val macroF1={best:.4f}")
    return va, lo, em

set_seed()
folds = sorted(meta.fold.unique())
if FAST: folds = folds[:1]
oof=[]; emb=[]; emb_id=[]
for k in folds:
    va, lo, em = train_fold(k); pr = softmax(lo)
    for i,(_,r) in enumerate(va.iterrows()):
        oof.append(dict(file_id=r.file_id, y=int(r.y), pred=int(pr[i].argmax()),
                        max_prob=float(pr[i].max()),
                        **{f"prob_{j}":float(pr[i,j]) for j in range(len(CLASSES))}))
        emb.append(em[i]); emb_id.append(r.file_id)
oof=pd.DataFrame(oof); oof.to_csv(OUT/"oof_cnn.csv", index=False)
np.save(OUT/"oof_embed.npy", np.asarray(emb, np.float32)); json.dump(emb_id, open(OUT/"oof_embed_ids.json","w"))
print("\nOOF 3-class macro-F1 =", round(f1_score(oof.y, oof.pred, average='macro'),4))
for i,c in enumerate(CLASSES):
    print(f"  recall[{c}] =", round(recall_score(oof.y, oof.pred, labels=[i], average=None)[0],4))

## 8. Unknown threshold — leave-one-class-out feasibility + operating τ

We hold each species out entirely, train on the other two, and measure whether
a low max-softmax flags the held-out species as novel (AUROC). This estimates
how detectable an unseen species is, and sets τ, **without the test set**.

In [ ]:
from sklearn.metrics import roc_auc_score
def loco_tau(seed=SEED, epochs=(10 if not FAST else 2)):
    aur=[]; by={}
    for held in range(len(CLASSES)):
        keep=[c for c in range(len(CLASSES)) if c!=held]; remap={c:i for i,c in enumerate(keep)}
        tr=meta[meta.y.isin(keep)].copy(); tr["y"]=tr.y.map(remap)
        vfold=int(meta.fold.mode().iloc[0])
        fit=tr[tr.fold!=vfold].reset_index(drop=True); kv=tr[tr.fold==vfold].reset_index(drop=True)
        uv=meta[meta.y==held].reset_index(drop=True)
        nb=NoiseBank(noise_bank[noise_bank.source_group.isin(set(fit.source_group))])
        cnt=np.bincount(fit.y.values,minlength=len(keep)).astype(float)
        sm=WeightedRandomSampler(torch.as_tensor(1.0/cnt[fit.y.values],dtype=torch.double),len(fit),True)
        dl=DataLoader(BirdDS(fit,True,nb,seed), BATCH, sampler=sm, num_workers=2, drop_last=True)
        m=build_model(n=len(keep)).to(device); opt=torch.optim.AdamW(m.parameters(),lr=2e-3,weight_decay=1e-4)
        sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=2e-3,total_steps=max(1,len(dl))*epochs,pct_start=0.15)
        crit=nn.CrossEntropyLoss(label_smoothing=0.05); scaler=torch.amp.GradScaler("cuda",enabled=device.type=="cuda")
        m.train()
        for _ in range(epochs):
            for X,y in dl:
                X=X.to(device);y=y.to(device);opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda",enabled=device.type=="cuda"): loss=crit(m(X),y)
                scaler.scale(loss).backward();scaler.step(opt);scaler.update();sch.step()
        @torch.no_grad()
        def conf(tbl):
            dl2=DataLoader(BirdDS(tbl.assign(y=0),False,None,seed),BATCH,num_workers=2); o=[]
            for X,_ in dl2: o.append(softmax(m(X.to(device)).float().cpu().numpy()))
            return np.concatenate(o)
        pk,pu=conf(kv),conf(uv)
        sc=np.concatenate([1-pk.max(1),1-pu.max(1)]); is_u=np.concatenate([np.zeros(len(pk)),np.ones(len(pu))])
        a=roc_auc_score(is_u,sc); aur.append(a)
        by[CLASSES[held]]=dict(auroc=round(float(a),3), known_maxp=round(float(np.median(pk.max(1))),3),
                               unk_maxp=round(float(np.median(pu.max(1))),3))
        print(f"  held={CLASSES[held]:12s} AUROC={a:.3f} known~{np.median(pk.max(1)):.2f} unk~{np.median(pu.max(1)):.2f}")
    # operating tau: midpoint between typical known and unknown confidence
    taus=[ (by[c]['known_maxp']+by[c]['unk_maxp'])/2 for c in CLASSES ]
    return float(np.median(taus)), dict(by_class=by, mean_auroc=round(float(np.mean(aur)),3))

set_seed()
tau, loco = loco_tau()
json.dump({"tau":tau,"loco":loco}, open(OUT/"unknown_threshold.json","w"), indent=2)
print("\nmean novelty AUROC:", loco["mean_auroc"], "| chosen tau:", round(tau,3))

## 9. Submission template — same format as `Arif_01_Submission.csv`

The result file is written in exactly the template of `Arif_01_Submission.csv`:
header `file_id,label`; one row per test clip in `sample_submission.csv` order;
labels limited to the three species + `Unknown`; comma-separated, no index, no
quotes, no BOM; **CRLF** line endings (also after the last row).
`write_submission()` enforces this and re-checks the saved file. Bump
`SUBMISSION_NO` in the config for every new upload.

In [ ]:
# Template = Arif_01_Submission.csv: header `file_id,label`; one row per test clip in
# sample_submission order; labels in {3 species, Unknown}; comma-separated, no index,
# no quotes, no BOM; CRLF line endings, including after the last row.
import shutil

SUB_NAME   = f"{AUTHOR}_{SUBMISSION_NO:02d}_Submission.csv"
SUB_COLS   = ["file_id", "label"]
SUB_LABELS = set(CLASSES) | {UNKNOWN}
SUB_EOL    = "\r\n"

def write_submission(out, sub):
    out = out[SUB_COLS]                                  # exact columns, exact order
    assert out.label.notna().all(), "some test clips have no prediction"
    assert out.file_id.tolist() == sub.file_id.tolist(), "file_id values/order differ from sample_submission.csv"
    bad = sorted(set(out.label) - SUB_LABELS)
    assert not bad, f"labels outside the template: {bad}"
    path = OUT/SUB_NAME
    # to_csv ends lines with LF on Linux/Kaggle but CRLF on Windows: normalise to CRLF and
    # write bytes, so Windows can't turn CRLF into CR-CR-LF.
    txt = out.to_csv(index=False).replace("\r\n", "\n").replace("\n", SUB_EOL)
    path.write_bytes(txt.encode("utf-8"))
    # re-read the saved file and check that it really is the template
    raw = path.read_bytes()
    assert raw.startswith((",".join(SUB_COLS) + SUB_EOL).encode()), "header differs from template"
    assert raw.endswith(SUB_EOL.encode()) and raw.count(b"\n") == raw.count(b"\r\n") == len(out) + 1, \
        "line endings differ from template"
    back = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert back.values.tolist() == out.astype(str).values.tolist(), "saved rows differ from predictions"
    return path

## 10. Inference → submission file

In [ ]:
def load_models():
    ms=[]
    for p in sorted(OUT.glob("model_fold*.pt")):
        c=torch.load(p, map_location=device, weights_only=False)
        m=build_model(size=c["size"]).to(device); m.load_state_dict(c["state_dict"]); m.eval(); ms.append(m)
    return ms
models=load_models(); print("ensemble size:", len(models), "| tau:", round(tau,3))

@torch.no_grad()
def predict_clip(y, max_w=12):
    wins=eval_windows(y)
    if len(wins)>max_w:
        idx=np.linspace(0,len(wins)-1,max_w).round().astype(int); wins=[wins[i] for i in idx]
    X=torch.as_tensor(np.stack([frontend(w)[None] for w in wins]).astype(np.float32), device=device)
    P=np.zeros((len(wins),len(CLASSES)))
    for m in models:
        with torch.amp.autocast("cuda", enabled=device.type=="cuda"): P+=softmax(m(X).float().cpu().numpy())
    return (P/len(models)).mean(0)

sub=pd.read_csv(SAMPLE_SUB); rows=[]
for n,fid in enumerate(sub.file_id):
    ap=CACHE/"audio"/"test"/f"{fid}.npy"
    y=np.load(ap) if ap.exists() else load_mono_32k(TEST_AUDIO/f"{fid}.flac")
    p=predict_clip(y); mp=float(p.max())
    rows.append(dict(file_id=fid, label=(UNKNOWN if mp<tau else CLASSES[int(p.argmax())]),
                     max_prob=mp, **{f"prob_{j}":float(p[j]) for j in range(len(CLASSES))}))
    if (n+1)%300==0: print(f"  {n+1}/{len(sub)}")
pred=pd.DataFrame(rows); pred.to_csv(OUT/"test_probs.csv", index=False)
out=sub[["file_id"]].merge(pred[["file_id","label"]], on="file_id", how="left")
path=write_submission(out, sub)                 # Arif_NN_Submission.csv, template format, checked
shutil.copyfile(path, OUT/"submission.csv")     # identical copy under the default Kaggle name
print("\nlabel distribution:\n", out.label.value_counts().to_string())
print("\nwrote", path, "(+ identical submission.csv)", f"({len(out)} rows)")
out.head()

## Notes for the writeup / rules compliance
- **No external data, no pretrained weights.** The CNN is initialised randomly
  (`BirdCNN`) and learns only from the attached competition audio.
- **Reproducible.** One `SEED` seeds Python, numpy and torch; the fold split is
  chosen deterministically by seed search and saved to `folds.csv`.
- **Explainable.** Every step (front-end, folds, augmentation, loss, Unknown
  rule) is plain code in this notebook — no AutoML, no black-box submission.
- **Tune before submitting.** Set `FAST=False`, run top to bottom. Adjust `τ`
  only via the leave-one-class-out cell, never by peeking at the leaderboard.
- **Expected CV:** on held-out groups the 3-class macro-F1 lands ~0.80–0.86
  depending on the fold split (Cendrawasih is the fragile class, 4 groups only).
- **Submission format.** The result is saved as `{AUTHOR}_{SUBMISSION_NO:02d}_Submission.csv`
  (default `Arif_01_Submission.csv`) in the reference template — header `file_id,label`,
  `sample_submission.csv` row order, labels = 3 species + `Unknown`, CRLF line endings —
  and `write_submission()` verifies this before the notebook finishes.